In [4]:
from models import load_model
from torchvision.transforms import v2 
import torch 
from PIL import Image 

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model_name = 'UNet.th'

model = load_model(input_channels=3, name=model_name)
model.eval()
img_name = 'rio-amazonas2.jpg'
img = Image.open(f'./{img_name}').convert('RGB')

transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Resize((224, 224))
])

img_tensor = transform(img)

img_tensor = img_tensor.unsqueeze(0) # anadir dimension del bache [1, 3, 224, 224]

with torch.no_grad():
    output = model(img_tensor)
    pred_img = torch.sigmoid(output) > 0.5
    im = pred_img[0].flatten().cpu().numpy().astype('uint8')*255 # [0-255]
    im = im.reshape((224, 224))
    im = Image.fromarray(im)
    im.save(f'mask_pred_{model_name}_{img_name}')

